# 00 — Persiapan dan Verifikasi (V4 SVDB, Revisi 12)

Notebook ini dijalankan **sekali sebelum** notebook 01–03. Isinya:
1. cek lingkungan dan integritas (hash protokol dan metode);
2. informasi dataset untuk paper;
3. pembuatan cache embedding ImageNet beku untuk ketiga backbone (sekitar 0,5 jam per backbone);
4. pembekuan sampel XAI, dilakukan sebelum ada model yang dijelaskan.

**Aturan yang mengikat:**
- **Metode terkunci.** Metode Revisi 12 terkunci sejak holdout pertama dibuka (`outputs/holdout/opening_lock.json`). Jangan mengubah `metode_v4*.yaml`, `v4_pipeline.py`, `v4_s3.py`, maupun berkas lain yang masuk hash. Bila ada yang berubah, notebook akan berhenti dengan galat hash.
- **Holdout hanya untuk evaluasi akhir.** Holdout dievaluasi satu kali per model final dan tidak dipakai untuk memilih apa pun.
- **Sel aman dijalankan ulang.** Run yang sudah lengkap diverifikasi lalu dipakai ulang. Run yang terputus dilanjutkan dari `last.pt`. Log disalin ke `outputs/logs/`.
- **Jangan hapus atau timpa output.** Bila perlu memulai ulang, pindahkan foldernya ke arsip.


In [ ]:
import sys, json, time
from pathlib import Path
FOLDER = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "v4_pipeline.py").exists() or (p / "eksperimen_v4_svdb" / "v4_pipeline.py").exists())
FOLDER = FOLDER if (FOLDER / "v4_pipeline.py").exists() else FOLDER / "eksperimen_v4_svdb"
sys.path.insert(0, str(FOLDER))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
import v4_runner as R
import v4_figures as F
print("Folder eksperimen:", FOLDER)


## 1. Lingkungan


In [ ]:
import platform, torch, shap
env = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2) if torch.cuda.is_available() else None,
       "shap": shap.__version__, "cpu": platform.processor(), "os": platform.platform()}
p = R.protocol()   # verifikasi protokol data + kunci metode S1/S2 dan S3
(p.out / "env.json").write_text(json.dumps(env, indent=2)); env


## 2. Integritas: hash protokol, metode, dan kunci holdout
Bila salah satu hash berbeda dari nilai yang tercatat, **berhenti** dan cari tahu penyebabnya. Jangan mengubah file lock.


In [ ]:
import v4_pipeline as v4, v4_s3
EXPECTED = {"method_hash": "5a131004958b", "s3_method_hash": "8bb5f1d88a8a"}
current = {"data_protocol_hash": p.hash, "method_hash": v4.method_hash(), "s3_method_hash": v4_s3.s3_method_hash()}
for key, prefix in EXPECTED.items():
    assert current[key].startswith(prefix), f"{key} berubah: {current[key]}"
lock = p.out / "holdout" / "opening_lock.json"
if lock.exists():
    opened = json.loads(lock.read_text())
    assert {k: opened[k] for k in current} == current, "Metode berbeda dari saat holdout pertama dibuka."
    print("Holdout pertama dibuka:", opened["first_opened"])
pd.Series(current)


## 3. Informasi dataset
Semua angka dihitung ulang dari manifest dan dicocokkan dengan nilai yang diharapkan di konfigurasi. Tabel disimpan ke `outputs/paper/` (CSV/LaTeX).


In [ ]:
from ecgtl.hierarchical.data import read_manifest
cfg = p.cfg["data"]
raw = read_manifest(p.prepared / "global_manifest.csv")
dev, hold = F.manifest("development"), F.manifest("holdout")
checks = pd.DataFrame([
    ("window manifest", len(raw), cfg["expected_windows"]),
    ("window development (manifest)", int((raw.split == "development").sum()), cfg["expected_development_windows"]),
    ("window holdout (manifest)", int((raw.split == "holdout").sum()), cfg["expected_holdout_windows"]),
    ("pasien total", raw.patient_id.nunique(), cfg["expected_patients"]),
    ("pasien development", dev.patient_id.nunique(), cfg["expected_development_patients"]),
    ("pasien holdout", hold.patient_id.nunique(), cfg["expected_holdout_patients"]),
], columns=["item", "aktual", "diharapkan"])
assert (checks.aktual == checks.diharapkan).all(), checks
assert not set(dev.patient_id) & set(hold.patient_id), "Overlap pasien development-holdout"
assert dev.groupby("patient_id").fold.nunique().max() == 1, "Pasien muncul di lebih dari satu fold"
display(checks)
print(f"Setelah validitas label (Revisi 9): development {len(dev):,} window, holdout {len(hold):,} window")


In [ ]:
tables = F.dataset_tables()
display(Markdown("**Development: window per kelas × database**")); display(tables["development"])
display(Markdown("**Holdout: window per kelas × database**")); display(tables["holdout"])
display(Markdown("**Distribusi fold development**")); display(tables["folds"])
counts = dev.label.value_counts().reindex(F.CLASSES)
print("Rasio ketidakseimbangan (terbanyak/tersedikit):", round(counts.max() / counts.min(), 2))
print("Proporsi kelas development (%):", (100 * counts / counts.sum()).round(1).to_dict())


In [ ]:
# Window yang dikeluarkan aturan validitas label, distribusi window per pasien, dan jumlah crop lokal per window
excluded = raw[~raw.window_id.isin(pd.concat([dev, hold]).window_id)]
display(Markdown("**Window yang dikeluarkan (Revisi 9: N hanya dari mitdb/svdb)**"))
display(pd.crosstab([excluded.split, excluded.label], excluded.database))
per_patient = pd.concat([dev, hold]).groupby(["split", "patient_id"]).size()
display(Markdown("**Window per pasien**")); display(per_patient.groupby(level=0).describe().round(1))
local = read_manifest(p.prepared / "local_manifest.csv")
crops = local[local.parent_window_id.isin(pd.concat([dev, hold]).window_id)].groupby("parent_window_id").size()
display(Markdown("**Crop lokal (beat) per window**")); display(crops.describe().round(2))
pd.DataFrame({"tabel": ["window_per_pasien", "crop_per_window"],
              "median": [per_patient.median(), crops.median()]}).to_csv(p.out / "paper" / "A02_dataset_ringkas.csv", index=False)


In [ ]:
# Pasien per kelas per fold, pasien S per sumber, dan bobot kelas (tau=1 / tau=2) per fold
display(Markdown("**Pasien per kelas per fold**")); display(dev.groupby(["fold", "label"]).patient_id.nunique().unstack()[F.CLASSES])
display(Markdown("**Pasien S per sumber per fold**")); display(dev[dev.label == "S"].groupby(["fold", "database"]).patient_id.nunique().unstack())
exp = R.experiment("efficientnet_b0", "S1"); rows = []
for fold in range(5):
    train = exp.dev[exp.dev.fold != fold]
    for tau in (1.0, 2.0):
        weights, _ = exp.base.weights(train, tau)
        rows.append({"fold": fold, "tau": tau, **dict(zip(F.CLASSES, weights["final"].cpu().numpy().round(3)))})
weights = pd.DataFrame(rows); weights.to_csv(p.out / "paper" / "A02_bobot_kelas.csv", index=False)
display(Markdown("**Bobot kelas loss akhir (tau=1 = S1, tau=2 = S2/S3)**")); display(weights)


In [ ]:
F.fig_dataset(); F.fig_examples(); F.fig_flow(); plt.show()


## 4. Cache embedding ImageNet beku (S1/S2)
Cache dibuat sekali per backbone (sekitar 0,5 jam). Bila sudah ada, sel ini hanya memverifikasi checksum-nya.


In [ ]:
for arch in R.ARCHS:
    t = time.time(); print(arch, R.experiment(arch, "S1").base.cache_features().to_dict("records"), f"{time.time() - t:.0f} dtk")


## 5. Pembekuan sampel XAI
Sampel dipilih **dari label saja**, tanpa melihat prediksi model mana pun, lalu dibekukan dengan hash. Komposisinya:
- 10 window holdout per kelas, dengan pasien sebanyak mungkin berbeda; 2 window pertama per kelas menjadi galeri;
- background 4 window development per kelas.

Sampel yang sama dipakai untuk ke-9 model.


In [ ]:
import v4_xai as X
samples = X.freeze_samples()
display(pd.DataFrame({c: [len(samples["explain"][c]), ", ".join(samples["gallery"][c]), len(samples["background"][c])] for c in F.CLASSES},
                     index=["dijelaskan", "galeri", "background"]))
print("sha256:", samples["sha256"])


In [ ]:
R.status()
